In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [6]:
# 读取 Kaggle 测试集与提交样例
df = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
submission = pd.read_csv("../data/sample_submission.csv")

# 查看数据规模
print("训练集：", df.shape)
print("测试集：", test.shape)
print("提交样例：", submission.shape)

# 比较字段
print("\n训练集字段：", df.columns.tolist())
print("\n测试集字段：", test.columns.tolist())
print("\n提交字段：", submission.columns.tolist())

训练集： (594194, 21)
测试集： (254655, 20)
提交样例： (254655, 2)

训练集字段： ['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

测试集字段： ['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

提交字段： ['id', 'Churn']


In [7]:
# 1. 定义特征与目标变量
X = df.drop(columns=["id", "Churn"])
y = (df["Churn"] == "Yes").astype(int)

# 2. 分层划分训练集与验证集
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. 查看划分结果
print("训练集：", X_train.shape)
print("验证集：", X_valid.shape)

print(f"训练集流失率：{y_train.mean():.2%}")
print(f"验证集流失率：{y_valid.mean():.2%}")

训练集： (475355, 19)
验证集： (118839, 19)
训练集流失率：22.52%
验证集流失率：22.52%


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 连续数值特征
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

# 类别特征（包含二元变量）
categorical_features = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

# SeniorCitizen 已经是 0/1，直接保留
binary_features = ["SeniorCitizen"]

# 构建预处理器
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features),
        ("binary", "passthrough", binary_features)
    ]
)

print("连续变量：", numeric_features)
print("类别变量数量：", len(categorical_features))
print("二元变量：", binary_features)

连续变量： ['tenure', 'MonthlyCharges', 'TotalCharges']
类别变量数量： 15
二元变量： ['SeniorCitizen']


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 构建逻辑回归流水线
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# 训练模型
lr_pipeline.fit(X_train, y_train)

# 预测验证集客户的流失概率
y_valid_proba = lr_pipeline.predict_proba(X_valid)[:, 1]

# 计算 ROC-AUC
auc = roc_auc_score(y_valid, y_valid_proba)

print(f"Logistic Regression ROC-AUC: {auc:.4f}")

Logistic Regression ROC-AUC: 0.9084


In [10]:
# 训练集预测概率
y_train_proba = lr_pipeline.predict_proba(X_train)[:, 1]

# 计算训练集与验证集 ROC-AUC
train_auc = roc_auc_score(y_train, y_train_proba)
valid_auc = roc_auc_score(y_valid, y_valid_proba)

print(f"训练集 ROC-AUC：{train_auc:.4f}")
print(f"验证集 ROC-AUC：{valid_auc:.4f}")
print(f"AUC 差值：{train_auc - valid_auc:.4f}")

训练集 ROC-AUC：0.9078
验证集 ROC-AUC：0.9084
AUC 差值：-0.0006


In [11]:
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 以 0.5 为分类阈值
y_valid_pred = (y_valid_proba >= 0.5).astype(int)

# 计算评价指标
print(f"ROC-AUC: {valid_auc:.4f}")
print(f"PR-AUC (AP): {average_precision_score(y_valid, y_valid_proba):.4f}")
print(f"Precision: {precision_score(y_valid, y_valid_pred):.4f}")
print(f"Recall: {recall_score(y_valid, y_valid_pred):.4f}")
print(f"F1: {f1_score(y_valid, y_valid_pred):.4f}")

# 混淆矩阵
print("\n混淆矩阵：")
print(confusion_matrix(y_valid, y_valid_pred))

ROC-AUC: 0.9084
PR-AUC (AP): 0.7271
Precision: 0.6839
Recall: 0.6577
F1: 0.6705

混淆矩阵：
[[83941  8135]
 [ 9162 17601]]


In [12]:
from sklearn.metrics import precision_score, recall_score, f1_score

# 设置不同分类阈值
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

results = []

for threshold in thresholds:
    # 根据阈值生成预测标签
    y_pred = (y_valid_proba >= threshold).astype(int)

    # 计算评价指标
    results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_valid, y_pred),
        "Recall": recall_score(y_valid, y_pred),
        "F1": f1_score(y_valid, y_pred),
        "Predicted Churn": y_pred.sum()
    })

# 整理结果
threshold_df = pd.DataFrame(results)

threshold_df.round(4)

,Threshold,Precision,Recall,F1,Predicted Churn
0,0.2,0.5317,0.8935,0.6667,44974
1,0.3,0.5831,0.8311,0.6854,38142
2,0.4,0.6310,0.7566,0.6881,32092
3,0.5,0.6839,0.6577,0.6705,25736
4,0.6,0.7426,0.5211,0.6125,18781
5,0.7,0.8185,0.3300,0.4704,10791


In [13]:
import numpy as np
import pandas as pd

# 按预测流失概率从高到低排序
ranking = pd.DataFrame({
    "y_true": np.asarray(y_valid),
    "risk_score": y_valid_proba
}).sort_values("risk_score", ascending=False)

total_churn = ranking["y_true"].sum()
n = len(ranking)

results = []

for budget_ratio in [0.1, 0.2, 0.3]:
    k = int(n * budget_ratio)

    # 选择风险最高的 K 名客户
    selected = ranking.head(k)

    tp = selected["y_true"].sum()

    results.append({
        "Budget Ratio": budget_ratio,
        "Contacted": k,
        "True Churn": tp,
        "Precision@K": tp / k,
        "Recall@K": tp / total_churn
    })

budget_df = pd.DataFrame(results)

budget_df.round(4)

,Budget Ratio,Contacted,True Churn,Precision@K,Recall@K
0,0.1,11883,9542,0.8030,0.3565
1,0.2,23767,16626,0.6995,0.6212
2,0.3,35651,21514,0.6035,0.8039
